# Consensus-labelling report

Aggregates saved label tables into diagnostic summaries and figures for the pseudo-labelling stage.


In [1]:
# ==========================================
# 1. LOAD AND DEDUPLICATE
# ==========================================
import glob
import os

import numpy as np
import pandas as pd

ROOT = r"M:\Github\TFM_SeismicPhasePicker_ColombianAndesRegion"
LAB_DIR = os.path.join(ROOT, "data", "processed", "labeling")
UNL_DIR = os.path.join(ROOT, "data", "processed", "unlabeling")
INV_DIR = os.path.join(ROOT, "data", "interim", "preprocessing")
OUT_DIR = os.path.join(ROOT, "data", "processed", "eval_results", "labeling_report")
os.makedirs(OUT_DIR, exist_ok=True)

SNR_CUT_DB = 0.0  # per-phase SNR QC threshold used in production

lab = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(LAB_DIR, "labels_*.csv")))], ignore_index=True)
unl = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(UNL_DIR, "unlabelled_*.csv")))], ignore_index=True)
raw = pd.concat([lab, unl], ignore_index=True)

# Remove duplicate trace names before computing report totals.
df = raw.drop_duplicates(subset="trace_name").reset_index(drop=True)
N = len(df)

# Cross-check against preprocessing inventories (status == ok).
inv_ok = 0
for fp in sorted(glob.glob(os.path.join(INV_DIR, "window_inventory_*.csv"))):
    inv_ok += int((pd.read_csv(fp)["status"] == "ok").sum())

print(f"raw rows (with duplicates) : {len(raw):,}")
print(f"unique windows (deduped)   : {N:,}")
print(f"inventory 'ok' windows     : {inv_ok:,}")
assert N == inv_ok, "deduped windows should equal inventory ok windows"

eq = df[df.category == "earthquake"].copy()
noise = df[df.category == "noise"].copy()
unlab = df[df.category == "unlabelled"].copy()
print(f"\nearthquake={len(eq):,}  noise={len(noise):,}  unlabelled={len(unlab):,}")

def pct(x, tot):
    return round(100.0 * x / tot, 1)

raw rows (with duplicates) : 316,478
unique windows (deduped)   : 191,488
inventory 'ok' windows     : 191,488

earthquake=53,861  noise=12,637  unlabelled=124,990


## R1 - Category split over the 191 488 valid windows
Source: column `category`. Absolute value and percentage (1 decimal); the three add up to 191 488.

In [2]:
r1 = []
for c in ["earthquake", "noise", "unlabelled"]:
    n = int((df.category == c).sum())
    r1.append({"category": c, "n": n, "%": pct(n, N)})
r1 = pd.DataFrame(r1)
print(r1.to_string(index=False))
print(f"\nsum = {int(r1.n.sum()):,}  (must be {N:,})   %sum = {r1['%'].sum():.1f}")

  category      n    %
earthquake  53861 28.1
     noise  12637  6.6
unlabelled 124990 65.3

sum = 191,488  (must be 191,488)   %sum = 100.0


## R2 - Accepted P and S labels (after SNR QC) + earthquake breakdown
Source: `p_sample` / `s_sample` not null on `earthquake` rows. Groups both / only-P / only-S add up to the earthquake total.

In [3]:
P_acc = int(eq.p_sample.notna().sum())
S_acc = int(eq.s_sample.notna().sum())
both = int((eq.p_sample.notna() & eq.s_sample.notna()).sum())
only_p = int((eq.p_sample.notna() & eq.s_sample.isna()).sum())
only_s = int((eq.p_sample.isna() & eq.s_sample.notna()).sum())
print(f"Accepted P labels : {P_acc:,}")
print(f"Accepted S labels : {S_acc:,}")
print(f"\nEarthquakes with both phases : {both:,}  ({pct(both, len(eq))}%)")
print(f"Earthquakes with only P      : {only_p:,}  ({pct(only_p, len(eq))}%)")
print(f"Earthquakes with only S      : {only_s:,}  ({pct(only_s, len(eq))}%)")
print(f"sum = {both + only_p + only_s:,}  (earthquakes = {len(eq):,})")

Accepted P labels : 49,910
Accepted S labels : 37,075

Earthquakes with both phases : 33,124  (61.5%)
Earthquakes with only P      : 16,786  (31.2%)
Earthquakes with only S      : 3,951  (7.3%)
sum = 53,861  (earthquakes = 53,861)


## R3 - Vote distribution
Source: `p_n_models` (on accepted P) and `s_n_models` (on accepted S). P rule = 3/4; S rule = 2/4.

In [4]:
vp = eq.loc[eq.p_sample.notna(), "p_n_models"].astype(int).value_counts().sort_index()
vs = eq.loc[eq.s_sample.notna(), "s_n_models"].astype(int).value_counts().sort_index()
print("P votes (within accepted P = {:,}):".format(P_acc))
for k, v in vp.items():
    print(f"  {k}/4 : {int(v):,}  ({pct(int(v), P_acc)}%)")
print("\nS votes (within accepted S = {:,}):".format(S_acc))
for k, v in vs.items():
    print(f"  {k}/4 : {int(v):,}  ({pct(int(v), S_acc)}%)")

P votes (within accepted P = 49,910):
  3/4 : 36,570  (73.3%)
  4/4 : 13,340  (26.7%)

S votes (within accepted S = 37,075):
  2/4 : 9,936  (26.8%)
  3/4 : 10,064  (27.1%)
  4/4 : 17,075  (46.1%)


## R4 - Labelling rate per station (22 stations)
Source: grouping `category` by `station_code`. Sorted by rate, descending.

In [5]:
r4 = df.groupby("station_code").agg(
    valid_windows=("category", "size"),
    earthquakes=("category", lambda s: int((s == "earthquake").sum())),
)
r4["rate_%"] = (100 * r4.earthquakes / r4.valid_windows).round(1)
r4 = r4.sort_values("rate_%", ascending=False)
print(r4.to_string())
print(f"\nstations = {len(r4)}   total valid = {int(r4.valid_windows.sum()):,}   total eq = {int(r4.earthquakes.sum()):,}")
r4.to_csv(os.path.join(OUT_DIR, "R4_rate_per_station.csv"))

              valid_windows  earthquakes  rate_%
station_code                                    
PRA                    9554         5203    54.5
ORTC                   9031         4721    52.3
RUS                    9931         4936    49.7
CBOC                  13585         5698    41.9
URE                    9130         3634    39.8
PTB                    9103         3120    34.3
GUY2C                 13067         4120    31.5
ZAR                    2034          626    30.8
CHI                    9767         2982    30.5
DBB                    5785         1665    28.8
YOT                    7153         2014    28.2
URMC                   4909         1282    26.1
JAMC                   5886         1263    21.5
PAL                   10683         2205    20.6
QUBC                   8503         1710    20.1
TAM                    4350          833    19.1
NOR                   15007         2601    17.3
PTGC                   6698         1086    16.2
BRJC                

## R5 - Labelling rate by magnitude interval
Source: `source_magnitude`. Half-magnitude bins; `n_windows` shows the bin size.

In [6]:
m_edges = np.arange(0.5, 7.0, 0.5)
df["_mbin"] = pd.cut(df.source_magnitude, m_edges, right=False)
r5 = df.groupby("_mbin", observed=True).agg(
    n_windows=("category", "size"),
    earthquakes=("category", lambda s: int((s == "earthquake").sum())),
)
r5["rate_%"] = (100 * r5.earthquakes / r5.n_windows).round(1)
print(r5.to_string())
r5.to_csv(os.path.join(OUT_DIR, "R5_rate_by_magnitude.csv"))

            n_windows  earthquakes  rate_%
_mbin                                     
[0.5, 1.0)        482          162    33.6
[1.0, 1.5)       3555         1079    30.4
[1.5, 2.0)     129290        27048    20.9
[2.0, 2.5)      40197        14849    36.9
[2.5, 3.0)      11773         6538    55.5
[3.0, 3.5)       3799         2473    65.1
[3.5, 4.0)       1542         1111    72.0
[4.0, 4.5)        573          411    71.7
[4.5, 5.0)        160          117    73.1
[5.0, 5.5)         64           41    64.1
[5.5, 6.0)         34           23    67.6
[6.0, 6.5)         19            9    47.4


## R6 - Labelling rate by epicentral distance interval
Source: `distance_km`. Intervals of 50 km up to 350 km.

In [7]:
d_edges = [0, 50, 100, 150, 200, 250, 300, 350]
df["_dbin"] = pd.cut(df.distance_km, d_edges, right=False)
r6 = df.groupby("_dbin", observed=True).agg(
    n_windows=("category", "size"),
    earthquakes=("category", lambda s: int((s == "earthquake").sum())),
)
r6["rate_%"] = (100 * r6.earthquakes / r6.n_windows).round(1)
print(r6.to_string())
r6.to_csv(os.path.join(OUT_DIR, "R6_rate_by_distance.csv"))

            n_windows  earthquakes  rate_%
_dbin                                     
[0, 50)           420          395    94.0
[50, 100)       15241        12317    80.8
[100, 150)      32022        16759    52.3
[150, 200)      30002        10285    34.3
[200, 250)      47798         8532    17.9
[250, 300)      42998         4025     9.4
[300, 350)      23007         1548     6.7


## R7 - P-S time separation (earthquakes with both phases)
Source: `s_time_s - p_time_s` on earthquakes with both `p_sample` and `s_sample`.

In [8]:
bothdf = eq[eq.p_sample.notna() & eq.s_sample.notna()].copy()
sp = (bothdf.s_time_s - bothdf.p_time_s).dropna()
print(f"n        = {len(sp):,}")
print(f"median   = {sp.median():.2f} s")
print(f"p10      = {sp.quantile(0.10):.2f} s")
print(f"p90      = {sp.quantile(0.90):.2f} s")
print(f"min      = {sp.min():.2f} s")
print(f"max      = {sp.max():.2f} s")
print(f"pairs < 1 s  : {int((sp < 1).sum()):,}")
print(f"pairs > 60 s : {int((sp > 60).sum()):,}")

n        = 33,124
median   = 15.60 s
p10      = 8.70 s
p90      = 27.84 s
min      = 0.01 s
max      = 113.82 s
pairs < 1 s  : 265
pairs > 60 s : 306


## R8 - SNR of accepted labels + SNR-QC drops
Source: `snr_p_db` / `snr_s_db`. A pick dropped by SNR keeps its `snr_*_db` but has
`sample = NaN`; `snr_*_db` is only computed when a consensus candidate existed, so
`sample NaN & snr not-null` identifies a consensus candidate rejected by SNR QC.

In [9]:
snP = eq.loc[eq.p_sample.notna(), "snr_p_db"].dropna()
snS = eq.loc[eq.s_sample.notna(), "snr_s_db"].dropna()
print("Accepted-label SNR (dB):")
print(f"  P : n={len(snP):,}  median={snP.median():.1f}  p10={snP.quantile(0.10):.1f}  p90={snP.quantile(0.90):.1f}")
print(f"  S : n={len(snS):,}  median={snS.median():.1f}  p10={snS.quantile(0.10):.1f}  p90={snS.quantile(0.90):.1f}")

# SNR-dropped candidates: sample NaN but SNR computed.
p_drop = df[df.p_sample.isna() & df.snr_p_db.notna()]
s_drop = df[df.s_sample.isna() & df.snr_s_db.notna()]
unl_by_snr = unlab[(unlab.snr_p_db.notna()) | (unlab.snr_s_db.notna())]
print("\nSNR-QC drops:")
print(f"  P phases dropped : {len(p_drop):,}"
      f"  (still earthquake via S: {int((p_drop.category=='earthquake').sum()):,},"
      f" became unlabelled: {int((p_drop.category=='unlabelled').sum()):,})")
print(f"  S phases dropped : {len(s_drop):,}")
print(f"  windows -> unlabelled because ALL phases were dropped by SNR : {len(unl_by_snr):,}")

print("\nConsistency check (R8 n vs R2 accepted):")
print(f"  P: accepted={P_acc:,}  SNR n={len(snP):,}  diff={P_acc - len(snP)}  (NaN SNR kept as accepted)")
print(f"  S: accepted={S_acc:,}  SNR n={len(snS):,}  diff={S_acc - len(snS)}")

Accepted-label SNR (dB):
  P : n=49,905  median=7.0  p10=1.5  p90=20.8
  S : n=37,072  median=7.5  p10=2.2  p90=13.7

SNR-QC drops:
  P phases dropped : 9,147  (still earthquake via S: 3,951, became unlabelled: 5,196)
  S phases dropped : 2,994
  windows -> unlabelled because ALL phases were dropped by SNR : 5,196

Consistency check (R8 n vs R2 accepted):
  P: accepted=49,910  SNR n=49,905  diff=5  (NaN SNR kept as accepted)
  S: accepted=37,075  SNR n=37,072  diff=3
